In [ ]:
import numpy as np
import pandas as pd

from __future__ import annotations
from typing import List, Set, Tuple
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

In [ ]:
class Tensor:
    def __init__(self, value: float, label: str = "", _prev: Set[Tensor] = set()):
        self.value = value
        self._prev = _prev
        self.grad = 0.0
        self._backward = lambda: None
        self.label = label if label != "" else str(value)

    def zero_grad(self):
        self.grad = 0.0

    def backward(self):
        # Topological Sort
        topo: List[Tensor] = []
        visited: Set[Tensor] = set()
        stack: List[Tuple[Tensor, bool]] = [(self, False)]

        while stack:
            node, processed = stack.pop()

            if processed:
                topo.append(node)
            elif node not in visited:
                visited.add(node)
                stack.append((node, True))
                for child in node._prev:
                    if child not in visited:
                        stack.append((child, False))

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __repr__(self) -> str:
        # return f"Tensor(label={self.label}, value={self.value}, grad={self.grad})"
        return f"{self.value}"

    def _repr_html_(self) -> str:
        """Renders an ultra-modern, dark-themed interactive card layout."""
        style = """
        <style>
            .dark-tensor-box {
                font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
                margin: 8px 0;
                background: #1e1e24;
                border: 1px solid #2d2d34;
                border-radius: 8px;
                overflow: hidden;
                max-width: 480px;
                box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2), 0 2px 4px -1px rgba(0, 0, 0, 0.14);
            }
            .dark-tensor-details[open] {
                border-bottom: 2px solid #6366f1;
            }
            .dark-tensor-summary {
                background: #25252d;
                padding: 10px 14px;
                cursor: pointer;
                user-select: none;
                font-weight: 600;
                color: #e2e8f0;
                font-size: 13px;
                display: flex;
                align-items: center;
                transition: background 0.2s ease;
            }
            .dark-tensor-summary:hover {
                background: #2a2a35;
            }
            .dark-tensor-summary::-webkit-details-marker {
                color: #6366f1;
                margin-right: 6px;
            }
            .dark-tensor-content {
                padding: 12px 16px;
                background: #19191e;
                font-size: 12px;
            }
            .dark-tensor-row {
                display: flex;
                justify-content: space-between;
                padding: 4px 0;
                border-bottom: 1px solid #24242b;
            }
            .dark-tensor-row:last-child {
                border-bottom: none;
            }
            .dark-tensor-key {
                color: #94a3b8;
            }
            .dark-tensor-val {
                font-weight: bold;
            }
            .val-label { color: #38bdf8; }   /* Cyber blue */
            .val-data  { color: #4ade80; }   /* Emerald green */
            .val-grad  { color: #f87171; }   /* Crimson red */
        </style>
        """

        display_label = self.label if self.label else "None"

        return f"""
        {style}
        <div class="dark-tensor-box">
            <details class="dark-tensor-details">
                <summary class="dark-tensor-summary">Tensor(label={display_label})</summary>
                <div class="dark-tensor-content">
                    <div class="dark-tensor-row">
                        <span class="dark-tensor-key">label</span>
                        <span class="dark-tensor-val val-label">"{display_label}"</span>
                    </div>
                    <div class="dark-tensor-row">
                        <span class="dark-tensor-key">value</span>
                        <span class="dark-tensor-val val-data">{self.value}</span>
                    </div>
                    <div class="dark-tensor-row">
                        <span class="dark-tensor-key">grad</span>
                        <span class="dark-tensor-val val-grad">{self.grad}</span>
                    </div>
                </div>
            </details>
        </div>
        """

    def __radd__(self, other: Tensor | float):
        return self + other

    def __add__(self, other: Tensor | float):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(
            self.value + other.value, f"({self.label} + {other.label})", {self, other}
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __neg__(self):
        out = self * Tensor(-1)
        return out

    def __sub__(self, other: Tensor | float):
        out = self + -other
        return out

    def __mul__(self, other: Tensor | float):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(
            self.value * other.value, f"{self.label} * {other.label}", {self, other}
        )

        def _backward():
            self.grad += out.grad * other.value
            other.grad += out.grad * self.value

        out._backward = _backward

        return out

    def __rmul__(self, other: Tensor | float):
        return self * other

    def __pow__(self, n: int):
        out = Tensor(
            self.value**n,
            f"{self.label} ** {n}",
            {self},
        )

        def _backward():
            self.grad += out.grad * n * self.value ** (n - 1)

        out._backward = _backward
        return out

    def __truediv__(self, other: Tensor | float):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self * other**-1

    def __gt__(self, other: Tensor):
        return self.value > other.value

    def __lt__(self, other: Tensor):
        return self.value < other.value

    def exp(self):
        out = Tensor(np.e**self.value, f"exp({self.label})", {self})

        def _backward():
            self.grad += out.grad * out.value

        out._backward = _backward
        return out

    def ln(self):
        self.value = 1e-15 if self.value == 0 else self.value
        out = Tensor(np.log(self.value), f"ln({self.label})", {self})

        def _backward():
            self.grad += out.grad * (1 / self.value)

        out._backward = _backward
        return out

    def relu(self):
        val = self.value if self.value > 0 else 0.0
        out = Tensor(val, f"relu({self.label})", {self})

        def _backward():
            self.grad += out.grad if self.value > 0 else 0.0

        out._backward = _backward
        return out

In [ ]:
class Neuron:
    def __init__(self, n_inputs) -> None:
        self.w = np.array([Tensor(np.random.randn() * .1) for _ in range(n_inputs)])
        self.b = Tensor(0.0)

    def __call__(self, x, activation=True):
        out = np.dot(self.w, x) + self.b
        if activation:
            out = out.relu()
        return out

    def get_params(self):
        return [*self.w, self.b]


class Layer:
    def __init__(self, n_inputs, n_units, name="") -> None:
        self.name = name
        self.neurons = np.array([Neuron(n_inputs) for _ in range(n_units)])

    def __call__(self, x, activation=True):
        return np.array([neuron(x, activation) for neuron in self.neurons])

    def __repr__(self) -> str:
        return f"Layer(name={self.name}, n_inputs={len(self.neurons[0].w)}, n_units={len(self.neurons)}"

    def get_params(self):
        params = []
        for neuron in self.neurons:
            params += neuron.get_params()
        return params


class MLP:
    def __init__(self, n_inputs, n_outputs) -> None:
        layer_sizes = [n_inputs] + n_outputs
        self.layers = np.array(
            [Layer(layer_sizes[i], layer_sizes[i + 1], f"L{i+1}") for i in range(len(n_outputs))]
        )

        params = []
        for layer in self.layers:
            params += layer.get_params()
        self.parameters = params

    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            is_last = (i == len(self.layers) - 1)
            x = layer(x, activation=not is_last)
        return x


In [ ]:
df = pd.read_csv("datasets/winequality-white.csv", sep=";")
input_scaler = StandardScaler()

X = df.iloc[:, :-1].to_numpy()
y = df.iloc[:, -1:].to_numpy().reshape(-1)

X_train = X[:100]
y_train = y[:100]

y_min, y_max = min(y_train), max(y_train)
n_outputs = y_max - y_min + 1

X_scaled_train = input_scaler.fit_transform(X_train)
y_scaled_train = y_train - y_min

X_scaled_test = input_scaler.transform(X[100:200])
y_scaled_test = y[100:200] - y_min

In [ ]:
mlp = MLP(11, [16, 16, n_outputs])
parameters = mlp.parameters
all_weights = [w for layer in mlp.layers for neuron in layer.neurons for w in neuron.w]

for i in tqdm(range(1000)):
    for parameter in parameters:
        parameter.zero_grad()

    total_loss = Tensor(0)

    for x, y_grd in zip(X_scaled_train, y_scaled_train):
        logits = mlp(x)

        # Softmax
        max_val = max(p.value for p in logits)
        exp_logits = [(p - max_val).exp() for p in logits]
        sum_exp = sum(exp_logits)
        probs = [exp_p / sum_exp for exp_p in exp_logits]

        # Cross Entropy
        correct_prob = probs[y_grd]
        sample_loss = -correct_prob.ln()
        total_loss += sample_loss

    total_loss = total_loss / len(X_scaled_train)

    # + L2 Regularization
    total_loss = total_loss + 0.001 * sum(w * w for w in all_weights)

    total_loss.backward()

    if ((i + 1) % 10) == 0:
        print(f"Epoch {i+1:>3}, Loss: {total_loss.value:>5.4f}")

    for parameter in parameters:
        parameter.value -= 0.1 * parameter.grad

In [ ]:
correct_test = 0
n_test = len(y_scaled_test)

for i in range(n_test):
    logits = mlp(X_scaled_test[i])

    y_pred = np.argmax(logits)

    if y_pred == y_scaled_test[i]:
        correct_test += 1

print(f"Test Accuracy: {correct_test / n_test:.2%}")


correct_train = 0
n_train = len(y_scaled_train)

for i in range(n_train):
    logits = mlp(X_scaled_train[i])
    y_pred = np.argmax(logits)

    if y_pred == y_scaled_train[i]:
        correct_train += 1

print(f"Train Accuracy: {correct_train / n_train:.2%}")